## **Previsão preventiva de insatisfação de clientes**
## **no e-commerce (Olist)**

## **Parte II - Engenharia de atributos e pré modelagem**

### 🎯 **Objetivo desta Etapa:**

#### nessa etapa, busca-se realizar os últimos tratamentos e transformações na base antes da modelagem, desenhando variáveis preditivas de alto impacto para o negócio para o modelo atuar como um **gatilho de monitoramento operacional (pós-venda)**. 

#### substituição dos dados do futuro por métricas honestas disponíveis nos primeiros dias logísticos do pedido, cobrindo o atraso de postagem do lojista, a janela consumida do frete, a fragilidade das categorias e a proporção de custo do frete. 

#### depois, tratamento das variáveis categóricas, subdivisão e alinhamento simétrico das partições de Treino Real, Validação Fixa e Teste.

## **Bibliotecas**

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import(
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    RandomizedSearchCV,
    GridSearchCV,
)

## **Base de dados**

In [3]:
# carregando a base
df = pd.read_pickle('../bases/base_tratada/dados_limpos.pkl')

print(f"Base carregada: {df.shape[0]} linhas | {df.shape[1]} colunas")

Base carregada: 94477 linhas | 32 colunas


## **Pré-modelagem**

### **Separação das bases**

In [4]:
# excluindo a base alvo
X = df.drop('avaliacao_ruim', axis=1)
y = df['avaliacao_ruim']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Tamanho Treino: {X_train.shape} | Tamanho Teste: {X_test.shape}")

Tamanho Treino: (75581, 31) | Tamanho Teste: (18896, 31)


## **features engineering**

### **prazo de entrega estimada**

In [5]:
# criando dias estimados para a entrega
X_train['dias_prometidos'] = (X_train['order_estimated_delivery_date'] - X_train['order_purchase_timestamp']).dt.days
X_test['dias_prometidos'] = (X_test['order_estimated_delivery_date'] - X_test['order_purchase_timestamp']).dt.days

Prazos de entrega muito longos (ex: 30 dias para cruzar o país) podem gerar ansiedade e deixar o cliente muito mais propenso a dar uma nota ruim se qualquer outro pequeno detalhe falhar.

### **Dias de atraso da postagem**

In [6]:
#Garantindo a conversão correta de todas as datas envolvidas
colunas_datas = [
    'order_purchase_timestamp', 'shipping_limit_date', 
    'order_delivered_carrier_date', 'order_estimated_delivery_date'
]
for col in colunas_datas:
    X_train[col] = pd.to_datetime(X_train[col])
    X_test[col]  = pd.to_datetime(X_test[col])

# =====================================================================
# DIAS DE ATRASO DO VENDEDOR NA POSTAGEM
# =====================================================================
# o vendedor postou antes ou depois do prazo limite do contrato?
X_train['dias_atraso_postagem'] = (X_train['order_delivered_carrier_date'] - X_train['shipping_limit_date']).dt.days
X_test['dias_atraso_postagem']  = (X_test['order_delivered_carrier_date'] - X_test['shipping_limit_date']).dt.days

# Se o resultado for negativo, significa que ele postou adiantado.
# Casos nulos (pedidos que ainda não tinham sido postados até o 5º dia) preenchemos com a mediana do treino.
mediana_atraso_postagem = X_train['dias_atraso_postagem'].median()
X_train['dias_atraso_postagem'] = X_train['dias_atraso_postagem'].fillna(mediana_atraso_postagem)
X_test['dias_atraso_postagem']  = X_test['dias_atraso_postagem'].fillna(mediana_atraso_postagem)


# =====================================================================
#  FLAG de Lojista postou com atraso?
# =====================================================================
# Se o atraso de postagem for maior que 0, o vendedor atrasou (1) , se nao (0)
X_train['flag_vendedor_atrasou'] = np.where(X_train['dias_atraso_postagem'] > 0, 1, 0)
X_test['flag_vendedor_atrasou']  = np.where(X_test['dias_atraso_postagem'] > 0, 1, 0)


# =====================================================================
# JANELA DO PRAZO PROMETIDO
# =====================================================================
# No 5º dia, o produto já foi postado? Quanto tempo sobrou até o prazo final estimado?
# Primeiro, se calcula quantos dias o cliente tem que esperar desde a compra até a transportadora coletar
X_train['dias_ate_postagem'] = (X_train['order_delivered_carrier_date'] - X_train['order_purchase_timestamp']).dt.days
X_test['dias_ate_postagem']  = (X_test['order_delivered_carrier_date'] - X_test['order_purchase_timestamp']).dt.days

mediana_ate_postagem = X_train['dias_ate_postagem'].median()
X_train['dias_ate_postagem'] = X_train['dias_ate_postagem'].fillna(mediana_ate_postagem)
X_test['dias_ate_postagem']  = X_test['dias_ate_postagem'].fillna(mediana_ate_postagem)

# Dias restantes prometidos ao cliente no momento em que o pacote sai para viajar:
# (Prazo Prometido menos os dias que o lojista gastou para postar)
X_train['dias_restantes_transporte'] = X_train['dias_prometidos'] - X_train['dias_ate_postagem']
X_test['dias_restantes_transporte']  = X_test['dias_prometidos'] - X_test['dias_ate_postagem']


### **dividindo a base por regiões** 

In [7]:
# criando regiões
mapeamento_regiao = {
    'SP': 'Sudeste', 'RJ': 'Sudeste', 'MG': 'Sudeste', 'ES': 'Sudeste',
    'PR': 'Sul', 'SC': 'Sul', 'RS': 'Sul',
    'BA': 'Nordeste', 'PE': 'Nordeste', 'CE': 'Nordeste', 'RN': 'Nordeste', 'PB': 'Nordeste', 'AL': 'Nordeste', 'SE': 'Nordeste', 'MA': 'Nordeste', 'PI': 'Nordeste',
    'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'DF': 'Centro-Oeste',
    'AM': 'Norte', 'PA': 'Norte', 'AC': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'AP': 'Norte', 'TO': 'Norte'
}

X_train['regiao'] = X_train['customer_state'].map(mapeamento_regiao)
X_test['regiao'] = X_test['customer_state'].map(mapeamento_regiao)

### **Criando flag para cliente e vendedor no mesmo estado e rota sudeste**

In [8]:
# Cliente e Vendedor estão no mesmo estado (1), Logística teoricamente rápida, se não (0)
X_train['mesmo_estado'] = np.where(X_train['customer_state'] == X_train['seller_state'], 1, 0)
X_test['mesmo_estado'] = np.where(X_test['customer_state'] == X_test['seller_state'], 1, 0)

# A compra aconteceu dentro da região Sudeste (1), Hub logístico principal do Olist, se não (0)
regiao_sudeste = ['SP', 'RJ', 'MG', 'ES']

X_train['rota_sudeste'] = np.where(X_train['customer_state'].isin(regiao_sudeste) & X_train['seller_state'].isin(regiao_sudeste), 1, 0)
X_test['rota_sudeste'] = np.where(X_test['customer_state'].isin(regiao_sudeste) & X_test['seller_state'].isin(regiao_sudeste), 1, 0)

### **Criando proporção do frete**

In [9]:
# Quanto o frete representa no custo total do pedido
# Usou-se o fillna(0) para evitar divisões por zero se o frete/valor for nulo
X_train['proporcao_frete'] = (X_train['valor_frete'] / X_train['valor_total']).fillna(0)
X_test['proporcao_frete']  = (X_test['valor_frete'] / X_test['valor_total']).fillna(0)

# Frete Abusivo (1) se frete custou mais que 30% do valor total pago, (0) se não.
X_train['frete_abusivo'] = np.where(X_train['proporcao_frete'] >= 0.30, 1, 0)
X_test['frete_abusivo'] = np.where(X_test['proporcao_frete'] >= 0.30, 1, 0)

### **Criando flags relacionadas ao produto**

In [10]:
# Criando flags para grandes blocos de categorias que historicamente quebram ou dão problemas
X_train['eletronico'] = np.where(
    X_train['product_category_name'].str.contains('eletronicos|informatica|telefonia', na=False, case=False), 1, 0)     # ignorando nulos ou variações dessas palavras


X_train['casa_decoracao'] = np.where(
    X_train['product_category_name'].str.contains('casa|moveis|decoracao|utilidades_domesticas', na=False, case=False), 1, 0)


X_train['perfumaria_beleza'] = np.where(
    X_train['product_category_name'].str.contains('perfumaria|beleza|saude', na=False, case=False), 1, 0)

# aplicando aos dados de teste
X_test['eletronico'] = np.where(
    X_test['product_category_name'].str.contains('eletronicos|informatica|telefonia', na=False, case=False), 1, 0)

X_test['casa_decoracao'] = np.where(
    X_test['product_category_name'].str.contains('casa|moveis|decoracao|utilidades_domesticas', na=False, case=False), 1, 0)

X_test['perfumaria_beleza'] = np.where(
    X_test['product_category_name'].str.contains('perfumaria|beleza|saude', na=False, case=False), 1, 0)

### **Mapeando perfil de recompra**

In [11]:
# ===================================
# FLAG DE CLIENTE RECORRENTE
# ===================================

# calculando a frequência histórica de cada cliente apenas na base de treino
df_frequencia_treino = X_train.groupby('customer_unique_id')['order_id'].nunique().reset_index()
df_frequencia_treino.columns = ['customer_unique_id', 'qtd_compras_treino']

# Transformando em um dicionário de clientes antigos
mapa_frequencia = df_frequencia_treino.set_index('customer_unique_id')['qtd_compras_treino'].to_dict()

# Mapeando a quantidade de compras para o X_train e X_test
# Se o cliente não for encontrado no mapa (novato), o fillna(1) assume que é a 1ª compra dele.
X_train['qtd_compras'] = X_train['customer_unique_id'].map(mapa_frequencia).fillna(1).astype(int)
X_test['qtd_compras']  = X_test['customer_unique_id'].map(mapa_frequencia).fillna(1).astype(int)

# 4. criando a Flag de Cliente Recorrente de forma segura em cada base
X_train['cliente_recorrente'] = np.where(X_train['qtd_compras'] > 1, 1, 0)
X_test['cliente_recorrente']  = np.where(X_test['qtd_compras'] > 1, 1, 0)

# Removendo a coluna auxiliar para não poluir o modelo
X_train = X_train.drop(columns=['qtd_compras'])
X_test  = X_test.drop(columns=['qtd_compras'])


### **Criando janela de postagem do vendedor**

In [12]:
X_train['shipping_limit_date'] = pd.to_datetime(X_train['shipping_limit_date'])
X_train['order_purchase_timestamp'] = pd.to_datetime(X_train['order_purchase_timestamp'])

X_test['shipping_limit_date'] = pd.to_datetime(X_test['shipping_limit_date'])
X_test['order_purchase_timestamp'] = pd.to_datetime(X_test['order_purchase_timestamp'])


# Quantos dias o vendedor tinha em contrato para postar o produto
X_train['dias_limite_postagem'] = (X_train['shipping_limit_date'] - X_train['order_purchase_timestamp']).dt.days
X_test['dias_limite_postagem']  = (X_test['shipping_limit_date'] - X_test['order_purchase_timestamp']).dt.days

# Calculando a mediana apenas no Treino
mediana_dias_treino = X_train['dias_limite_postagem'].median()

# Tratando possíveis erros de datas nulas ou inconsistentes preenchendo com a mediana
X_train['dias_limite_postagem'] = X_train['dias_limite_postagem'].fillna(mediana_dias_treino)
X_test['dias_limite_postagem']  = X_test['dias_limite_postagem'].fillna(mediana_dias_treino)

### **Criando faixa de valores**

In [13]:
# criando faixa de valores: 
#  0 = 'valor baixo'
#   1 = 'valor médio'
#   2 = 'valor alto' 

q1 = X_train['valor_total'].quantile(0.25)
q2 = X_train['valor_total'].quantile(0.75)

X_train['faixa_valor'] = np.where(X_train['valor_total'] < q1, 0, np.where(X_train['valor_total'] < q2, 1, 2))
X_test['faixa_valor'] = np.where(X_test['valor_total'] < q1, 0, np.where(X_test['valor_total'] < q2, 1, 2))

### **Transformando coluna 'mes_ano' em inteiro**

In [14]:
# Transformando o tipo 'period[M]' em NÚMERO INTEIRO (Ex: 2017-08 -> 201708)
# assim o modelo entende a ordem do tempo!
X_train['mes_ano'] = X_train['mes_ano'].dt.strftime('%Y%m').astype(int)
X_test['mes_ano'] = X_test['mes_ano'].dt.strftime('%Y%m').astype(int)

### **Criando volume do produto**

In [15]:
# criando volume do produto
X_train['volume_produto'] = X_train['product_length_cm'] * X_train['product_height_cm'] * X_train['product_width_cm']
X_test['volume_produto'] = X_test['product_length_cm'] * X_test['product_height_cm'] * X_test['product_width_cm']

In [16]:
# Lista de colunas do futuro ou redundantes
colunas_excluir= [
    # --- Dados do Futuro (Vazamento de Dados) ---
    'tempo_entrega_real', 'atraso_entrega', 'order_delivered_customer_date',
    
    # --- IDs e Textos Brutos (Quebram o Scikit-Learn) ---
    'order_id', 'customer_unique_id', 'seller_id',
    'product_category_name',
    'review_score',
    
    # --- Datas brutas que já viraram números ---
    'order_purchase_timestamp', 'order_estimated_delivery_date', 
    'shipping_limit_date', 'order_delivered_carrier_date',    
    'customer_state', # Já usou a região no get_dummies

    # --- outras não necessarias nas proximas etapas ---
    'seller_state', 'janela_postagem',
    'tempo_entrega_estimado',           # redundante, já tem a coluna 'dias prometidos'
    'flag_vendedor_atrasou'     # redundante
]

# Drop seguro aplicando no X_train e X_test
X_train = X_train.drop(columns=colunas_excluir, errors='ignore')
X_test  = X_test.drop(columns=colunas_excluir, errors='ignore')

print(f"✅ Limpeza concluída! estrutura atual: {X_train.shape} colunas numéricas.")


✅ Limpeza concluída! estrutura atual: (75581, 30) colunas numéricas.


## **Sub divisão da base de treino**

In [17]:
# Extraindo 20% do X_train original para criar a base de validação
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, 
    test_size=0.20, 
    random_state=42, 
    stratify=y_train # Garante a mesma proporção de notas ruins em ambas
)

# O y_test continua isolado e guardado desde o começo


### **Dummificação das colunas categóricas**

In [18]:
# Colunas que vão virar flags binárias
colunas_dummies = ['payment_type', 'regiao']

X_tr   = pd.get_dummies(X_tr,   columns=colunas_dummies, drop_first=False, dtype=int)
X_val  = pd.get_dummies(X_val,  columns=colunas_dummies, drop_first=False, dtype=int)
X_test = pd.get_dummies(X_test, columns=colunas_dummies, drop_first=False, dtype=int)

# garantindo as mesmas colunas em ambos, pois as vezes o teste tem uma categoria rara que o treino não tem, ou vice-versa.
colunas_corretas = X_tr.columns
X_val = X_val.reindex(columns=colunas_corretas, fill_value=0)
X_test = X_test.reindex(columns=colunas_corretas, fill_value=0)

In [19]:
print(f'base de treino: {X_tr.shape[0]} linhas | {X_tr.shape[1]} colunas')
print(f'base de validação: {X_val.shape[0]} linhas | {X_val.shape[1]} colunas')
print(f'base de teste: {X_test.shape[0]} linhas | {X_test.shape[1]} colunas')

base de treino: 60464 linhas | 37 colunas
base de validação: 15117 linhas | 37 colunas
base de teste: 18896 linhas | 37 colunas


In [20]:
# Salvando para modelagem
import joblib
joblib.dump((X_tr, X_val, X_test, y_tr, y_val, y_test), '../bases/bases_modelagem/dados_modelagem.pkl')
print("\n💾 Salvo em: bases/bases_modelagem/dados_modelagem.pkl")


💾 Salvo em: bases/bases_modelagem/dados_modelagem.pkl


## **Conclusões dessa etapa e próximos passos**

#### Nessa etapa foi estabelecido um ecossistema de dados maduro, robusto e alinhado aos objetivos da solução de negócio.

#### 📈 Principais entregas:
- **Blindagem contra Data Leakage:** Eliminou-se todas as variáveis de controle logístico do futuro (como datas e atrasos reais de entrega) e substituiu-se por variáveis de expectativa e histórico disponíveis até o momento da inferência do modelo no pós-venda.

- **Inclusão de Features Engineering:** Criou-se novas variáveis preditivas baseadas na jornada de compra, cobrindo média histórica do vendedor, desvio padrão das notas, impactos de frete, a fragilidade por categorias de produtos, o trajeto geopolítico e o perfil de recorrência do cliente.

- **Tratamento e Divisão Consistente:** Filtrou-se os registros não avaliados e segmentou-se os dados de forma estratificada em **Treino Real (X_tr)**, **Validação Fixa (X_val)** e 

**Teste Blindado (X_test)**, garantindo a simetria absoluta de colunas via técnica de `.reindex`.

O arquivo final de modelagem contendo as 6 partições foi exportado com sucesso no formato binário compacto (`dados_modelagem.pkl`).

---
**Próximo Passo:** carregar esta base, realizar o torneio de algoritmos e realizar ajustes para mitigar os falsos negativos e positivos!